In [13]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import re

from tqdm import trange, tqdm

x = np.load('x.npy')
x = np.clip(x, 0, 26)

y = np.load('y.npy')
y = np.clip(y, 0, 26)

In [14]:
GPU_indx = 0
device = torch.device(GPU_indx if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [15]:
LEN = len(x)
print(LEN)
split = int(0.8*LEN)

x_train = torch.tensor(x[:split])
y_train = torch.tensor(y[:split])
x_test = torch.tensor(x[split:])
y_test = torch.tensor(y[split:])

51010


In [16]:
class ShakeText(Dataset):
    start_index = 27
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        start  = torch.tensor([self.start_index], dtype=torch.long)
        target = torch.from_numpy(y[idx]).long()
        dec_in = torch.cat([start, target[:-1]], dim=0)
        return x[idx], dec_in, target
    
VOCAB_SIZE = 28
KEY_LEN = 5

In [17]:
batch_size = 64
train_loader = DataLoader(ShakeText(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(ShakeText(x_test, y_test), batch_size=batch_size, shuffle=False)

In [18]:
X, d, Y = next(iter(train_loader))

print(X.shape)
print(Y.shape)
print(d.shape)

torch.Size([64, 100])
torch.Size([64, 5])
torch.Size([64, 5])


In [20]:
bidirec=True

class Bidirectional_Encoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_size, hidden_size,
                            batch_first=True, bidirectional=True)

    def forward(self, x):
        x = self.embedding(x)
        outputs, (hn, cn) = self.lstm(x)
        
        hn_combined = torch.cat((hn[0, :, :], hn[1, :, :]), dim=1).unsqueeze(0)
        cn_combined = torch.cat((cn[0, :, :], cn[1, :, :]), dim=1).unsqueeze(0)
        
        return outputs, hn_combined, cn_combined

class Attention_Decoder(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_size + hidden_size, hidden_size, batch_first=True)
        self.fc   = nn.Linear(hidden_size, output_size)

    def forward(self, x, hidden_state, cell_state, encoder_outputs):
        
        x = self.embedding(x.unsqueeze(1))  
        hidden_reshaped = hidden_state.permute(1, 2, 0)
        
        raw_scores = torch.bmm(encoder_outputs, hidden_reshaped)
        attention_weights = F.softmax(raw_scores, dim=1)
        
        weights_flipped = attention_weights.permute(0, 2, 1)
        context = torch.bmm(weights_flipped, encoder_outputs)
        
        lstm_input = torch.cat((x, context), dim=2)
        
        out, (new_hidden, new_cell) = self.lstm(lstm_input, (hidden_state, cell_state))
        prediction = self.fc(out)
        
        return prediction, new_hidden, new_cell

In [21]:
class Model(nn.Module) :
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, ans, trainer=True):
        outputs, hn_combined, cn_combined = self.encoder(src)
        all_logits = []
        curr_token = ans[:, 0]
        for i in range(KEY_LEN):
            prediction, hn_combined, cn_combined = self.decoder(curr_token, hn_combined, cn_combined, outputs)
            all_logits.append(prediction)
            if trainer and i<KEY_LEN-1:
                curr_token = ans[:, i+1]
            else:
                curr_token = prediction.squeeze(1).argmax(dim=-1)
        return torch.cat(all_logits, dim=1)

In [ ]:
EMBED_SIZE   = 64
ENC_HIDDEN   = 256
DEC_HIDDEN   = ENC_HIDDEN * 2   

encoder = Bidirectional_Encoder(
    vocab_size  = VOCAB_SIZE,
    embed_size  = EMBED_SIZE,
    hidden_size = ENC_HIDDEN,
    pad_idx     = 0
)

decoder = Attention_Decoder(
    vocab_size  = VOCAB_SIZE,
    embed_size  = EMBED_SIZE,
    hidden_size = DEC_HIDDEN,
    output_size = VOCAB_SIZE,
    pad_idx     = 0
)

model = Model(encoder, decoder).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
n_epochs = 50
print(model)

Model(
  (encoder): Bidirectional_Encoder(
    (embedding): Embedding(28, 64, padding_idx=0)
    (lstm): LSTM(64, 256, batch_first=True, bidirectional=True)
  )
  (decoder): Attention_Decoder(
    (embedding): Embedding(28, 64, padding_idx=0)
    (lstm): LSTM(576, 512, batch_first=True)
    (fc): Linear(in_features=512, out_features=28, bias=True)
  )
)


In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = []

    for cipher, dec_in, key in tqdm(loader, desc='Train', leave=False):
        cipher = cipher.to(device)
        dec_in = dec_in.to(device)
        key    = key.to(device)

        optimizer.zero_grad()
        logits = model(cipher, dec_in, trainer=True)
        loss   = criterion(logits.permute(0, 2, 1), key)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss.append(loss.item())

    return running_loss


def test_epoch(model, loader, criterion):
    model.eval()
    running_loss = []
    correct = 0
    total   = 0

    with torch.no_grad():
        for cipher, dec_in, key in tqdm(loader, desc='Test', leave=False):
            cipher = cipher.to(device)
            dec_in = dec_in.to(device)
            key    = key.to(device)

            logits = model(cipher, dec_in, trainer=False)
            loss   = criterion(logits.permute(0, 2, 1), key)
            running_loss.append(loss.item())

            preds   = logits.argmax(dim=-1)
            correct += (preds == key).sum().item()
            total   += key.numel()

    acc = 100.0 * correct / total
    return running_loss, acc

In [24]:
train_loss_log = []
test_loss_log  = []
test_acc_log   = []

for epoch in trange(n_epochs, desc='Epoch'):
    tl = train_epoch(model, train_loader, criterion, optimizer)
    vl, acc = test_epoch(model, test_loader, criterion)

    train_loss_log.extend(tl)
    test_loss_log.extend(vl)
    test_acc_log.append(acc)

    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch+1:3d}/{n_epochs}  '
              f'train_loss={np.mean(tl):.4f}  '
              f'val_loss={np.mean(vl):.4f}  '
              f'acc={acc:.2f}%')

print(f'\nFinal test accuracy: {test_acc_log[-1]:.2f}%')

Epoch:   4%|▍         | 2/50 [01:23<33:33, 41.95s/it]

Epoch   2/50  train_loss=2.6659  val_loss=2.7932  acc=15.09%


Epoch:   8%|▊         | 4/50 [02:49<32:30, 42.40s/it]

Epoch   4/50  train_loss=2.5262  val_loss=2.7442  acc=15.70%


Epoch:  12%|█▏        | 6/50 [04:14<31:04, 42.37s/it]

...

Epoch  46/50  train_loss=0.4257  val_loss=1.1968  acc=76.21%


Epoch:  96%|█████████▌| 48/50 [59:27<11:57, 358.72s/it]

Epoch  48/50  train_loss=0.3804  val_loss=1.0936  acc=78.80%


Epoch: 100%|██████████| 50/50 [1:00:42<00:00, 72.85s/it] 

Epoch  50/50  train_loss=0.3398  val_loss=0.9449  acc=82.25%

Final test accuracy: 82.25%


In [ ]:
for epoch in trange(10, desc='Epoch'):
    tl = train_epoch(model, train_loader, criterion, optimizer)
    vl, acc = test_epoch(model, test_loader, criterion)

    train_loss_log.extend(tl)
    test_loss_log.extend(vl)
    test_acc_log.append(acc)

    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch+1:3d}/{n_epochs}  '
              f'train_loss={np.mean(tl):.4f}  '
              f'val_loss={np.mean(vl):.4f}  '
              f'acc={acc:.2f}%')

print(f'\nFinal test accuracy: {test_acc_log[-1]:.2f}%')

Epoch:  10%|█         | 2/20 [01:13<11:05, 36.95s/it]

Epoch   2/50  train_loss=0.3064  val_loss=0.8184  acc=85.30%


Epoch:  20%|██        | 4/20 [02:28<09:54, 37.15s/it]

Epoch   4/50  train_loss=0.2739  val_loss=0.7392  acc=87.08%


Epoch:  30%|███       | 6/20 [03:43<08:40, 37.20s/it]

...

Epoch  16/50  train_loss=0.1531  val_loss=0.2915  acc=95.93%


Epoch:  90%|█████████ | 18/20 [11:13<01:14, 37.08s/it]

Epoch  18/50  train_loss=0.1403  val_loss=0.2867  acc=95.88%


Epoch: 100%|██████████| 20/20 [12:27<00:00, 37.37s/it]

Epoch  20/50  train_loss=0.1288  val_loss=0.2339  acc=96.73%

Final test accuracy: 96.73%


In [26]:
for epoch in trange(10, desc='Epoch'):
    tl = train_epoch(model, train_loader, criterion, optimizer)
    vl, acc = test_epoch(model, test_loader, criterion)

    train_loss_log.extend(tl)
    test_loss_log.extend(vl)
    test_acc_log.append(acc)

    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch+1:3d}/{n_epochs}  '
              f'train_loss={np.mean(tl):.4f}  '
              f'val_loss={np.mean(vl):.4f}  '
              f'acc={acc:.2f}%')

print(f'\nFinal test accuracy: {test_acc_log[-1]:.2f}%')

Epoch:  20%|██        | 2/10 [01:30<06:00, 45.03s/it]

Epoch   2/50  train_loss=0.1198  val_loss=0.2152  acc=97.12%


Epoch:  40%|████      | 4/10 [03:09<04:46, 47.82s/it]

Epoch   4/50  train_loss=0.1140  val_loss=0.2058  acc=97.25%


Epoch:  60%|██████    | 6/10 [04:44<03:10, 47.70s/it]

Epoch   6/50  train_loss=0.1070  val_loss=0.2085  acc=97.16%


Epoch:  80%|████████  | 8/10 [06:25<01:38, 49.13s/it]

Epoch   8/50  train_loss=0.1003  val_loss=0.1929  acc=97.40%


Epoch: 100%|██████████| 10/10 [08:04<00:00, 48.48s/it]

Epoch  10/50  train_loss=0.0972  val_loss=0.1898  acc=97.40%

Final test accuracy: 97.40%


In [ ]:
def idx_to_char(idx):
    if idx == 0:  return ' '
    if 1 <= idx <= 26: return chr(idx - 1 + ord('a'))
    return '?'

model.eval()
with torch.no_grad():
    cipher_b, dec_in_b, key_b = next(iter(test_loader))
    logits = model(cipher_b.to(device), dec_in_b.to(device), trainer=False)
    preds  = logits.argmax(dim=-1).cpu()

print('Predicted  |  True')
print('-' * 23)
for i in range(10):
    pred_str = ''.join(idx_to_char(p.item()) for p in preds[i])
    true_str = ''.join(idx_to_char(k.item()) for k in key_b[i])
    match = '✓' if pred_str == true_str else ' '
    print(f'{pred_str}      |  {true_str}  {match}')

Predicted  |  True
-----------------------
insin      |  insin  ✓
xierv      |  xierv  ✓
thdil      |  thdil  ✓
nenji      |  nenji  ✓
dzgdr      |  dzgdr  ✓
tpmpr      |  tpmpr  ✓
zfeyv      |  zfeyv  ✓
xdsle      |  xdsle  ✓
dxdqv      |  dxdqv  ✓
miegw      |  miegw  ✓
